## Load Wandb database to csv

In [1]:
import pickle
from tqdm.notebook import tqdm  # tqdm for Jupyter notebooks

import pandas as pd
import wandb
from omegaconf import OmegaConf
import pandas as pd
import json  # if the dict is in string format
from pandas import json_normalize

entity = "Long_Memory_Arena"
api = wandb.Api()  # reads WANDB_API_KEY from env or ~/.netrc
projects = [p.name for p in api.projects(entity=entity)]

In [2]:
def retrive_data(project):
    print(f'Retriving {project}')
    api = wandb.Api()

    # Project is specified by <entity/project-name>
    runs = api.runs(f"Long_Memory_Arena/{project}")

    summary_list, config_list, name_list, tag_list = [], [], [], []
    loss_list = []

    for run in tqdm(runs):
        # .summary contains the output keys/values for metrics like accuracy.
        #  We call ._json_dict to omit large files
        summary_list.append(run.summary._json_dict)

        # .config contains the hyperparameters.
        #  We remove special values that start with _.
        config_list.append({k: v for k, v in run.config.items() if not k.startswith("_")})

        # .name is the human-readable name of the run.
        name_list.append(run.name)

        tag_list.append(run.tags)

        try:
            history = run.scan_history(keys=["train/rLoss1_epoch"])
            losses = [row["train/rLoss1_epoch"] for row in history if "train/rLoss1_epoch" in row]
        except Exception as e:
            print(f"Failed to get loss for run {run.name}: {e}")
            losses = []

        loss_list.append(losses)


    df = pd.DataFrame(
        {"summary": summary_list, "config": config_list, "name": name_list, "tag": tag_list, "loss_history": loss_list}
    )

    with open(f"{project}_data.pkl", "wb") as f:
        pickle.dump(df, f)
    print(f'Saved {project}')

def clean_data(p):
    with open(f"{p}_data.pkl", "rb") as f:
        df = pickle.load(f)

    # Example: Assuming `df` is your DataFrame, and the column is called 'config'
    # If it's a string, convert to actual dict:
    df['config_dict'] = df['config'].apply(lambda x: eval(x) if isinstance(x, str) else x)
    df['summary_dict'] = df['summary'].apply(lambda x: eval(x) if isinstance(x, str) else x)
    # Now flatten the nested dict into separate columns
    expanded_df = json_normalize(df['config_dict'])
    expanded_df1 = json_normalize(df['summary_dict'])

    # Merge back with original df if needed
    df = df.drop(columns=['config'])  # optionally drop the original column
    df = df.drop(columns=['summary'])
    df = pd.concat([df, expanded_df,expanded_df1], axis=1)

    df.to_pickle(f"{p}_data_clean.pkl")
    

# Retrive Data

In [7]:
projects = [p.name for p in api.projects(entity=entity) if "step3" in p.name]
projects

['Transformer_polynomial_step3',
 'Transformer_airy_step3',
 'Transformer_impulse_step3',
 'Transformer_exponential_step3',
 'TCN_impulse_step3',
 'TCN_airy_step3',
 'S4_airy_step3',
 'TCN_poly_step3',
 'S4_impulse_step3',
 'S4_exp_step3',
 'LSTM_impulse_step3',
 'LSTM_airy_step3',
 'TCN_exp_step3',
 'S4_poly_step3',
 'LSTM_poly_step3',
 'LSTM_exp_step3']

In [8]:
for p in projects:
    retrive_data(p)

    clean_data(p)
    print(f'Cleaned {p}')

Retriving Transformer_polynomial_step3


  0%|          | 0/364 [00:00<?, ?it/s]

Saved Transformer_polynomial_step3
Cleaned Transformer_polynomial_step3
Retriving Transformer_airy_step3


  0%|          | 0/343 [00:00<?, ?it/s]

Saved Transformer_airy_step3
Cleaned Transformer_airy_step3
Retriving Transformer_impulse_step3


  0%|          | 0/350 [00:00<?, ?it/s]

... (2 outputs left out)

wandb: WARNING A graphql request initiated by the public wandb API timed out (timeout=19 sec). Create a new API with an integer timeout larger than 19, e.g., `api = wandb.Api(timeout=29)` to increase the graphql timeout.


Saved Transformer_exponential_step3
Cleaned Transformer_exponential_step3
Retriving TCN_impulse_step3


  0%|          | 0/224 [00:00<?, ?it/s]

KeyboardInterrupt: 